# Day 15 · Lab 4/4: hard patterns, performance tuning and a mock exam

The last lab:
1. Four patterns behind Zillacode's Hard problems: sessions, streaks, the first and last row per group, the latest version from several snapshots, and comparing with the whole table
2. Spark SQL inside `etl`, and why SQL and DataFrame code run the same plan
3. 🔬 Read less: columns, sorted files, partition folders and small files
4. 🔬 Caching: when computing once and reusing pays, and when it hurts
5. UDFs: when you need one, and what it costs (measured)
6. Debugging, and the performance mistakes that give no error
7. ⚡ The slow report: a final optimization problem that uses all three labs
8. **Mock exam**: 6 Zillacode-style problems in 75 minutes

On the optimization side this is theory Days 4, 7 and 8: file layout and pushdown, partition pruning, small files, caching, UDF costs and the habits of fast Spark code. The notebook ends with a one-page optimization checklist, then the Zillacode setup (the last task of Day 15 in the roadmap).

## How to work
* Run the four **Setup** cells first, then the cell that prints the small tables. If you ran Day 14 in the same folder, the big tables are already in `data/big/` and are only read.
* **Lessons**: run every example cell (its output is already shown under it), then change it and run it again. Times will differ on your machine; the plans, tasks and bytes should not.
* Exercises: **`etl(...)`** in Zillacode's format, **🔍 Read the plan** (answer as a comment), **⚡ Make it faster** (the expected output *and* the target under it).
* Keep the **Spark UI** open (normally http://localhost:4040). Today you'll also use its **Storage** tab (cached data).
* The answers are collapsed at the very end of the notebook. Open one only after you have tried. The mock exam has no hints: do it with a timer.
* Today: about 1 hour 30 minutes of lessons, 1 hour 15 minutes of exercises (4 `etl`, 3 🔍, 3 ⚡) and the 75-minute mock exam. It's the longest day; if it doesn't fit, take the mock exam the next morning, before you start Zillacode.

**Setup (once):** Java 17 or 21 and Python 3.10 or newer. In a new folder: `python3.12 -m venv .venv`, `source .venv/bin/activate`, `pip install pyspark==4.2.0 jupyterlab`, then `jupyter lab` and open this notebook. Lesson 3 writes three more copies of `sales` into `data/big/` (about 450 MB in all); delete the `data/` folder when you're done.

## The data
The small shop tables of Days 12 to 14 (saved as Parquet in `data/shop/`), for the `etl` exercises, and the big generated tables of Day 14 for the 🔬 lessons.

* **customers**: *customer_id*, *name*, *email*, *city*, *signup_date*
* **products**: *product_id*, *product_name*, *category*, *price* (dollars for one item)
* **orders**: *order_id*, *customer_id*, *product_id*, *quantity*, *order_date*, *status* (`delivered`, `shipped` or `cancelled`)
* **reviews**: *review_id*, *product_id*, *customer_id*, *review_date* (text like `15/01/2024`), *rating* (1 to 5), *comment* (messy text), *tags* (text like `fast delivery;good price`)
* **signups_raw**: sign-ups exactly as a web form sent them: *signup_id*, *full_name*, *email*, *phone*, *signup_date* (text in three formats), *amount* (text), *prefs* (JSON text)
* **customers_new**: the customers table one week later
* **promotions**: *promo_id*, *category*, *start_date*, *end_date* (both days included), *discount_pct*
* **events**: website visits: *customer_id*, *event_time* (a timestamp), *page*
* **referrals**: *referrer_id* (the customer who invited someone), *referred_id* (the customer who joined), *referral_date*

The data has the traps that Zillacode's hidden tests like to use: order 1011 is a guest checkout (no *customer_id*); customer 6 has no email and no city; customer 7 has never ordered; product 107 has never been sold; order 1016 is for product 108, which is not in *products*; review 6 has no tags; two events were loaded twice; referral 5 points to customer 9, who doesn't exist.

**The big tables** (third setup cell), for the 🔬 lessons:
* **sales**: 10 million rows: *sale_id*, *customer_id*, *product_id*, *store_id* (1 to 200), *quantity* (1 to 5), *sale_date* (2023-01-01 to 2024-12-31), *channel* (`web`, `app` or `store`). About 140 MB.
* **customers_big**: 2 million customers: *customer_id*, *name*, *city* (10 cities), *segment* (`business` or `private`). About 18 MB, just above Spark's 10 MB broadcast limit, on purpose.
* **products_big**: 50,000 products: *product_id*, *product_name*, *category* (10 categories), *price*. Under 1 MB.

**Customer 1 is a reseller who makes 30% of all sales** (about 3 million rows). Every other customer has a handful. Day 14, Lesson 5 showed what this does to a join.

## Setup
Run these cells once: Spark and the small tables, `measure()` (Day 13, Lesson 1), and the big tables of Day 14 (generated only if `data/big/` doesn't exist yet).

In [ ]:
# Setup: starts Spark, creates the tables and saves them as Parquet files. Run it once.
import logging, os, sys
from datetime import date
os.environ.setdefault("PYSPARK_PYTHON", sys.executable)   # Spark's Python workers use this notebook's Python
from pyspark.sql import SparkSession, functions as F, Window as W   # the same names as in Zillacode's editor

spark = (SparkSession.builder
         .master("local[4]")                               # 4 cores: up to 4 tasks run at the same time
         .appName("spark-lab-15")
         .config("spark.driver.memory", "2g")              # the default 1 GB is tight for 10 million rows
         .config("spark.sql.ansi.enabled", "false")        # behave like Zillacode (Spark 3): bad text gives null, not an error
         .config("spark.sql.session.timeZone", "UTC")
         .config("spark.driver.host", "localhost")         # run on this machine only; the Spark UI is at http://localhost:4040
         .config("spark.sql.maxMetadataStringLength", "1000")   # explain() prints long filter and column lists in full
         .config("spark.hadoop.parquet.hadoop.vectored.io.enabled", "false")   # Spark 4.2: without this, the UI's "Input" misses most bytes read
         .config("spark.ui.showConsoleProgress", "false")
         .config("spark.log.level", "FATAL")               # hide Spark's log messages
         # Spark 4.2.0 bug (SPARK-60294): Python UDFs on Parquet can hang without this
         .config("spark.sql.execution.arrow.pythonUDF.columnarInput.enabled", "false")
         .getOrCreate())
for name in ("DataFrameQueryContextLogger", "SQLQueryContextLogger"):
    logging.getLogger(name).setLevel(logging.CRITICAL)        # errors still show, without an extra JSON log

customers = spark.createDataFrame([
    # customer_id, name, email, city, signup_date
    (1, "Aziz Karimov",     "aziz@mail.uz",     "Tashkent", date(2024, 1, 5)),
    (2, "Bella Schmidt",    "bella@web.de",     "Berlin",   date(2024, 1, 20)),
    (3, "Carlos Ruiz",      "carlos@correo.es", "Madrid",   date(2024, 2, 2)),
    (4, "Dilnoza Rahimova", "dilnoza@mail.uz",  "Tashkent", date(2024, 2, 14)),
    (5, "Emma Wilson",      "emma@post.co.uk",  "London",   date(2024, 3, 1)),
    (6, "Farid Aliyev",     None,               None,       date(2024, 3, 10)),
    (7, "Grace Lee",        "grace@web.de",     "Berlin",   date(2024, 3, 15)),
], "customer_id INT, name STRING, email STRING, city STRING, signup_date DATE")

products = spark.createDataFrame([
    # product_id, product_name, category, price (dollars for one item)
    (101, "Laptop",       "Electronics", 1200.0),
    (102, "Headphones",   "Electronics",  150.0),
    (103, "Coffee Maker", "Kitchen",       85.0),
    (104, "Blender",      "Kitchen",       60.0),
    (105, "Novel",        "Books",         15.0),
    (106, "Cookbook",     "Books",         25.0),
    (107, "Desk Lamp",    "Home",          40.0),
], "product_id INT, product_name STRING, category STRING, price DOUBLE")

orders = spark.createDataFrame([
    # order_id, customer_id, product_id, quantity, order_date, status
    (1001, 1,    101, 1, date(2024, 1, 10), "delivered"),
    (1002, 1,    105, 3, date(2024, 1, 10), "delivered"),
    (1003, 2,    102, 2, date(2024, 1, 25), "delivered"),
    (1004, 3,    103, 1, date(2024, 2, 3),  "cancelled"),
    (1005, 2,    106, 1, date(2024, 2, 5),  "delivered"),
    (1006, 4,    104, 2, date(2024, 2, 15), "shipped"),
    (1007, 1,    102, 1, date(2024, 2, 20), "delivered"),
    (1008, 5,    101, 1, date(2024, 3, 2),  "shipped"),
    (1009, 3,    105, 4, date(2024, 3, 5),  "delivered"),
    (1010, 4,    106, 2, date(2024, 3, 8),  "cancelled"),
    (1011, None, 104, 1, date(2024, 3, 9),  "delivered"),
    (1012, 6,    103, 2, date(2024, 3, 12), "shipped"),
    (1013, 2,    101, 1, date(2024, 3, 18), "delivered"),
    (1014, 5,    105, 5, date(2024, 3, 20), "delivered"),
    (1015, 1,    103, 1, date(2024, 3, 25), "shipped"),
    (1016, 3,    108, 1, date(2024, 3, 28), "delivered"),
], "order_id INT, customer_id INT, product_id INT, quantity INT, order_date DATE, status STRING")

reviews = spark.createDataFrame([
    # review_id, product_id, customer_id, review_date (text), rating, comment, tags (text)
    (1, 101, 1, "15/01/2024", 5, "  Great laptop, very FAST!  ",    "fast delivery;good price"),
    (2, 105, 1, "20/01/2024", 4, "Nice story.",                     "good price"),
    (3, 102, 2, "02/02/2024", 3, "Sound is ok, delivery was slow",  "slow delivery"),
    (4, 103, 3, "10/02/2024", 1, "Broke after one week!!",          "broken;slow delivery"),
    (5, 101, 5, "09/03/2024", 5, "Fast and powerful",               "fast delivery"),
    (6, 105, 3, "12/03/2024", 2, "Too long, boring",                None),
    (7, 106, 2, "25/03/2024", 4, "Useful recipes",                  "good price;gift"),
    (8, 103, 6, "30/03/2024", 4, "Good coffee, slow start",         "gift"),
], "review_id INT, product_id INT, customer_id INT, review_date STRING, rating INT, comment STRING, tags STRING")

signups_raw = spark.createDataFrame([
    # signup_id, full_name, email, phone, signup_date (text, three formats), amount (text), prefs (JSON text)
    (1, "  aziz   KARIMOV ", "Aziz@Mail.UZ ",      "+998 90 123-45-67", "2024-01-05", "12.50", '{"newsletter": true, "lang": "uz"}'),
    (2, "bella schmidt",     "bella@web.de",       "0049 151 2345678",  "20/01/2024", "abc",   '{"newsletter": false, "lang": "de"}'),
    (3, "Carlos  Ruiz",      "",                   "(34) 612 345 678",  "2024/02/02", "7",     '{"lang": "es"}'),
    (4, "DILNOZA rahimova",  "dilnoza@mail.uz",    "90-765-43-21",      "14/02/2024", "",      None),
    (5, "Emma Wilson",       "emma[at]post.co.uk", "07700 900123",      "2024-03-01", "3.75",  '{"newsletter": true}'),
    (6, "aziz karimov",      "aziz@mail.uz",       "+998901234567",     "2024-03-02", "1.00",  '{"newsletter": false, "lang": "uz"}'),
], "signup_id INT, full_name STRING, email STRING, phone STRING, signup_date STRING, amount STRING, prefs STRING")

customers_new = spark.createDataFrame([
    # the customers table one week later: 2 moved, 6 has an email now, 7 is gone, 8 is new
    (1, "Aziz Karimov",     "aziz@mail.uz",     "Tashkent", date(2024, 1, 5)),
    (2, "Bella Schmidt",    "bella@web.de",     "Munich",   date(2024, 1, 20)),
    (3, "Carlos Ruiz",      "carlos@correo.es", "Madrid",   date(2024, 2, 2)),
    (4, "Dilnoza Rahimova", "dilnoza@mail.uz",  "Tashkent", date(2024, 2, 14)),
    (5, "Emma Wilson",      "emma@post.co.uk",  "London",   date(2024, 3, 1)),
    (6, "Farid Aliyev",     "farid@mail.az",    None,       date(2024, 3, 10)),
    (8, "Hana Sato",        "hana@mail.jp",     "Osaka",    date(2024, 4, 2)),
], customers.schema)

promotions = spark.createDataFrame([
    # promo_id, category, start_date, end_date (both days included), discount_pct
    ("WINTER",   "Electronics", date(2024, 1, 1),  date(2024, 1, 31), 10),
    ("BOOKWEEK", "Books",       date(2024, 3, 1),  date(2024, 3, 10), 20),
    ("SPRING",   "Kitchen",     date(2024, 3, 1),  date(2024, 3, 31), 15),
    ("FLASH",    "Electronics", date(2024, 3, 15), date(2024, 3, 20), 5),
], "promo_id STRING, category STRING, start_date DATE, end_date DATE, discount_pct INT")

events = spark.createDataFrame([
    # customer_id, event_time, page (two rows were loaded twice)
    (1, "2024-03-01 09:00:00", "home"),
    (1, "2024-03-01 09:10:00", "product"),
    (1, "2024-03-01 09:10:00", "product"),
    (1, "2024-03-01 11:00:00", "home"),
    (1, "2024-03-02 10:00:00", "cart"),
    (1, "2024-03-03 10:30:00", "home"),
    (1, "2024-03-05 08:00:00", "home"),
    (2, "2024-03-01 20:00:00", "home"),
    (2, "2024-03-01 20:20:00", "product"),
    (2, "2024-03-01 20:45:00", "cart"),
    (2, "2024-03-04 18:00:00", "home"),
    (2, "2024-03-05 18:00:00", "home"),
    (2, "2024-03-05 18:00:00", "home"),
    (3, "2024-03-02 12:00:00", "product"),
    (3, "2024-03-03 12:00:00", "product"),
    (3, "2024-03-04 12:00:00", "cart"),
], "customer_id INT, event_time STRING, page STRING").withColumn("event_time", F.to_timestamp("event_time"))

referrals = spark.createDataFrame([
    # referrer_id (the customer who invited), referred_id (the customer who joined), referral_date
    (1, 4, date(2024, 2, 10)),
    (4, 5, date(2024, 2, 25)),
    (1, 6, date(2024, 3, 8)),
    (2, 7, date(2024, 3, 12)),
    (2, 9, date(2024, 3, 30)),
], "referrer_id INT, referred_id INT, referral_date DATE")

# Save every table as Parquet and read it back. In-memory tables would show "Scan ExistingRDD" in plans;
# Parquet files show what real jobs read: "FileScan parquet" with the columns and filters it reads.
for name, df in [("customers", customers), ("products", products), ("orders", orders), ("reviews", reviews),
                 ("signups_raw", signups_raw), ("customers_new", customers_new), ("promotions", promotions),
                 ("events", events), ("referrals", referrals)]:
    df.coalesce(1).write.mode("overwrite").parquet(f"data/shop/{name}")   # one file per table
customers = spark.read.parquet("data/shop/customers")
products = spark.read.parquet("data/shop/products")
orders = spark.read.parquet("data/shop/orders")
reviews = spark.read.parquet("data/shop/reviews")
signups_raw = spark.read.parquet("data/shop/signups_raw")
customers_new = spark.read.parquet("data/shop/customers_new")
promotions = spark.read.parquet("data/shop/promotions")
events = spark.read.parquet("data/shop/events")
referrals = spark.read.parquet("data/shop/referrals")

print("Spark", spark.version, "is ready. Spark UI:", spark.sparkContext.uiWebUrl)
print("Tables: customers, products, orders, reviews, signups_raw, customers_new, promotions, events, referrals")

Spark 4.2.0 is ready. Spark UI: http://localhost:4040
Tables: customers, products, orders, reviews, signups_raw, customers_new, promotions, events, referrals


In [ ]:
# measure(): run a query and print what it cost. These are the numbers the Spark UI shows.
import json, time, urllib.request

def ui(path):
    """Read one page of the Spark UI's REST API (the data behind http://localhost:4040)."""
    sc = spark.sparkContext
    with urllib.request.urlopen(f"{sc.uiWebUrl}/api/v1/applications/{sc.applicationId}/{path}") as response:
        return json.load(response)

def size(n):
    for unit, k in (("GB", 1e9), ("MB", 1e6), ("KB", 1e3)):
        if n >= k:
            return f"{n / k:.1f} {unit}"
    return f"{n} B"

def measure(query, label="", stages=False, plan=False):
    """Run `query` to the end, then print: time, jobs, stages, tasks, bytes read from files, shuffled and spilled.

    query:  a DataFrame (it runs with the "noop" writer: every task runs, nothing is kept or shown),
            or a function without arguments, like  lambda: df.count()
    stages: also print one line per stage: its tasks, the median and the slowest task, its shuffle
    plan:   also print the final plan of the query, after AQE's changes (the plan in the UI's SQL tab)
    """
    sc = spark.sparkContext
    group = f"measure-{time.time_ns()}"
    sc.setJobGroup(group, label or "measure")
    start = time.perf_counter()
    try:
        if callable(query):
            query()
        else:
            query.write.format("noop").mode("overwrite").save()
    finally:
        sc.setLocalProperty("spark.jobGroup.id", None)
    seconds = time.perf_counter() - start
    tracker = sc.statusTracker()
    jobs = sorted(tracker.getJobIdsForGroup(group))
    stage_ids = sorted({s for j in jobs for s in tracker.getJobInfo(j).stageIds})
    for _ in range(100):                     # the UI records a stage a moment after it ends
        info = [ui(f"stages/{s}")[0] for s in stage_ids]
        if all(s["status"] in ("COMPLETE", "SKIPPED", "FAILED") for s in info):
            break
        time.sleep(0.05)
    ran = [s for s in info if s["status"] == "COMPLETE"]       # "SKIPPED": output reused, did not run
    total = lambda key: sum(s[key] for s in ran)
    print(f"{label + ': ' if label else ''}{seconds:.2f} s | {len(jobs)} jobs, {len(ran)} stages, "
          f"{total('numCompleteTasks')} tasks | read {size(total('inputBytes'))} | "
          f"shuffle {size(total('shuffleWriteBytes'))} | spill {size(total('diskBytesSpilled'))}")
    if stages:
        for s in ran:
            t = ui(f"stages/{s['stageId']}/{s['attemptId']}/taskSummary?quantiles=0.5,1.0")["executorRunTime"]
            print(f"   stage {s['stageId']:>3}: {s['numCompleteTasks']:>3} tasks, median {t[0] / 1000:.2f} s, "
                  f"max {t[1] / 1000:.2f} s | read {size(s['inputBytes'])} | "
                  f"shuffle read {size(s['shuffleReadBytes'])}, write {size(s['shuffleWriteBytes'])}")
    if plan:
        runs = [e for e in ui("sql?offset=0&length=1000000") if set(e["successJobIds"]) & set(jobs)]
        tree = ui(f"sql/{runs[-1]['id']}?planDescription=true")["planDescription"].split("\n\n")[0]
        print(tree.replace(", Statistics(sizeInBytes=8.0 EiB)", ""))

print("measure() is ready")

measure() is ready


In [ ]:
# The big tables: generated once into data/big/ (about 15 seconds), then only read
import os
if not os.path.exists("data/big/sales/_SUCCESS"):
    spark.range(1, 10_000_001, numPartitions=8).select(
        F.col("id").alias("sale_id"),
        # customer 1 (a reseller) gets 30% of the sales; the rest are spread over customers 2 to 2,000,000
        F.when(F.rand(1) < 0.3, 1).otherwise((F.rand(2) * 1_999_999).cast("int") + 2).alias("customer_id"),
        ((F.rand(3) * 50_000).cast("int") + 1).alias("product_id"),
        ((F.rand(4) * 200).cast("int") + 1).alias("store_id"),
        ((F.rand(5) * 5).cast("int") + 1).alias("quantity"),
        F.date_add(F.lit("2023-01-01").cast("date"), (F.rand(6) * 731).cast("int")).alias("sale_date"),
        F.element_at(F.array(F.lit("web"), F.lit("app"), F.lit("store")), (F.rand(7) * 3).cast("int") + 1).alias("channel"),
    ).write.mode("overwrite").parquet("data/big/sales")
if not os.path.exists("data/big/customers/_SUCCESS"):
    cities = ["Tashkent", "Samarkand", "Bukhara", "Berlin", "Munich", "Madrid", "London", "Paris", "Rome", "Osaka"]
    spark.range(1, 2_000_001, numPartitions=4).select(
        F.col("id").cast("int").alias("customer_id"),
        F.concat(F.lit("Customer "), F.col("id")).alias("name"),
        F.element_at(F.array(*[F.lit(c) for c in cities]), (F.col("id") % 10).cast("int") + 1).alias("city"),
        F.when(F.col("id") % 7 == 0, "business").otherwise("private").alias("segment"),
    ).write.mode("overwrite").parquet("data/big/customers")
if not os.path.exists("data/big/products/_SUCCESS"):
    categories = ["Books", "Electronics", "Kitchen", "Home", "Toys", "Garden", "Sports", "Beauty", "Music", "Games"]
    spark.range(1, 50_001, numPartitions=1).select(
        F.col("id").cast("int").alias("product_id"),
        F.concat(F.lit("Product "), F.col("id")).alias("product_name"),
        F.element_at(F.array(*[F.lit(c) for c in categories]), (F.col("id") % 10).cast("int") + 1).alias("category"),
        F.round(F.rand(8) * 500 + 5, 2).alias("price"),
    ).write.mode("overwrite").parquet("data/big/products")

sales = spark.read.parquet("data/big/sales")
customers_big = spark.read.parquet("data/big/customers")
products_big = spark.read.parquet("data/big/products")
print(f"sales: {sales.count():,} rows | customers_big: {customers_big.count():,} | products_big: {products_big.count():,}")
sales.show(3)

sales: 10,000,000 rows | customers_big: 2,000,000 | products_big: 50,000
+-------+-----------+----------+--------+--------+----------+-------+
|sale_id|customer_id|product_id|store_id|quantity| sale_date|channel|
+-------+-----------+----------+--------+--------+----------+-------+
|2500001|          1|      1196|      15|       2|2023-12-20|  store|
|2500002|    1906291|      4755|     148|       3|2024-06-20|    app|
|2500003|     132273|     38526|      97|       4|2023-07-05|    web|
+-------+-----------+----------+--------+--------+----------+-------+
only showing top 3 rows


Print every table, so you can check results by eye:

In [ ]:
for name in ["customers", "products", "orders", "reviews", "signups_raw", "customers_new", "promotions", "events", "referrals"]:
    print(name)
    globals()[name].show(truncate=False)

customers
+-----------+----------------+----------------+--------+-----------+
|customer_id|name            |email           |city    |signup_date|
+-----------+----------------+----------------+--------+-----------+
|1          |Aziz Karimov    |aziz@mail.uz    |Tashkent|2024-01-05 |
|2          |Bella Schmidt   |bella@web.de    |Berlin  |2024-01-20 |
|3          |Carlos Ruiz     |carlos@correo.es|Madrid  |2024-02-02 |
|4          |Dilnoza Rahimova|dilnoza@mail.uz |Tashkent|2024-02-14 |
|5          |Emma Wilson     |emma@post.co.uk |London  |2024-03-01 |
|6          |Farid Aliyev    |NULL            |NULL    |2024-03-10 |
|7          |Grace Lee       |grace@web.de    |Berlin  |2024-03-15 |
+-----------+----------------+----------------+--------+-----------+

products
+----------+------------+-----------+------+
|product_id|product_name|category   |price |
+----------+------------+-----------+------+
|101       |Laptop      |Electronics|1200.0|
|102       |Headphones  |Electronics|150.

## Lesson 1 · Four patterns behind the Hard problems

Day 14 already covered top N per group, the latest row per key, running totals and growth. The Hard problems mostly add one of these four ideas.

### A. Sessions: split rows into groups wherever there's a gap
"A new session starts when more than 30 minutes passed since the previous event." The recipe:
1. Compute the gap to the previous row with `lag` (per user, ordered by time).
2. Make a flag column: 1 where a new group starts (the gap is too big, or there's no previous row), else 0.
3. A **running sum of the flag** numbers the groups: 1, 1, 1, 2, 2, 3...

```python
w = W.partitionBy("user").orderBy("time")
gap = F.unix_timestamp("time") - F.unix_timestamp(F.lag("time").over(w))           # seconds
flag = F.when(gap.isNull() | (gap > 30 * 60), 1).otherwise(0)
df.withColumn("new", flag).withColumn("session", F.sum("new").over(w.rowsBetween(W.unboundedPreceding, W.currentRow)))
```
After that, `groupBy(user, session)` gives each session's start, end, length and number of events.

In [ ]:
# Shopping trips: a new trip starts when more than 30 days passed since the customer's previous order
w = W.partitionBy("customer_id").orderBy("order_date", "order_id")
running = w.rowsBetween(W.unboundedPreceding, W.currentRow)
(orders.filter(F.col("customer_id").isin(1, 3))
       .withColumn("gap_days", F.datediff("order_date", F.lag("order_date").over(w)))
       .withColumn("new_trip", F.when(F.col("gap_days").isNull() | (F.col("gap_days") > 30), 1).otherwise(0))
       .withColumn("trip", F.sum("new_trip").over(running))
       .select("customer_id", "order_id", "order_date", "gap_days", "new_trip", "trip")
       .orderBy("customer_id", "order_date", "order_id")
       .show())

+-----------+--------+----------+--------+--------+----+
|customer_id|order_id|order_date|gap_days|new_trip|trip|
+-----------+--------+----------+--------+--------+----+
|          1|    1001|2024-01-10|    NULL|       1|   1|
|          1|    1002|2024-01-10|       0|       0|   1|
|          1|    1007|2024-02-20|      41|       1|   2|
|          1|    1015|2024-03-25|      34|       1|   3|
|          3|    1004|2024-02-03|    NULL|       1|   1|
|          3|    1009|2024-03-05|      31|       1|   2|
|          3|    1016|2024-03-28|      23|       0|   2|
+-----------+--------+----------+--------+--------+----+



### B. Streaks: runs of consecutive days ("gaps and islands")
"The longest run of consecutive days with a login." The trick: number each user's **distinct** days with `row_number`, then subtract that number from the date. Inside a run of consecutive days, the date goes up by 1 and the row number goes up by 1, so `day - row_number` stays **the same**. It changes as soon as a day is skipped. That constant (the "anchor") is a group key:

| day | row_number | day − row_number |
|---|---|---|
| 03-01 | 1 | 02-29 |
| 03-02 | 2 | 02-29 |
| 03-03 | 3 | 02-29 |
| 03-06 | 4 | 03-02 |
| 03-07 | 5 | 03-02 |

Then `groupBy(user, anchor)` gives each streak: `count` is its length, `min` and `max` its first and last day. Take the days `distinct` first: two logins on one day would break the arithmetic.

In [ ]:
logins = (spark.createDataFrame(
              [("ann", "2024-03-01"), ("ann", "2024-03-02"), ("ann", "2024-03-02"), ("ann", "2024-03-03"),
               ("ann", "2024-03-06"), ("ann", "2024-03-07"), ("bob", "2024-03-05")], "user string, day string")
          .withColumn("day", F.to_date("day"))
          .distinct())
w = W.partitionBy("user").orderBy("day")
islands = logins.withColumn("rn", F.row_number().over(w)).withColumn("anchor", F.date_sub("day", F.col("rn")))
islands.orderBy("user", "day").show()
(islands.groupBy("user", "anchor")
        .agg(F.min("day").alias("start"), F.max("day").alias("end"), F.count("*").alias("days"))
        .orderBy("user", "start")
        .show())

+----+----------+---+----------+
|user|       day| rn|    anchor|
+----+----------+---+----------+
| ann|2024-03-01|  1|2024-02-29|
| ann|2024-03-02|  2|2024-02-29|
| ann|2024-03-03|  3|2024-02-29|
| ann|2024-03-06|  4|2024-03-02|
| ann|2024-03-07|  5|2024-03-02|
| bob|2024-03-05|  1|2024-03-04|
+----+----------+---+----------+

+----+----------+----------+----------+----+
|user|    anchor|     start|       end|days|
+----+----------+----------+----------+----+
| ann|2024-02-29|2024-03-01|2024-03-03|   3|
| ann|2024-03-02|2024-03-06|2024-03-07|   2|
| bob|2024-03-04|2024-03-05|2024-03-05|   1|
+----+----------+----------+----------+----+



### C. The first and the last row of each group, in one `groupBy`
`F.min_by(value, ordering)` returns `value` from the row with the smallest `ordering`, and `F.max_by` from the row with the largest. Both work inside a normal `agg`, next to other aggregates. To break ties, order by a **struct**: `F.struct("order_date", "order_id")` compares the date first and the id second. The alternative is two `row_number` windows (one ascending, one descending) and a join, which is longer.

In [ ]:
by_time = F.struct("order_date", "order_id")
(orders.filter(F.col("customer_id").isNotNull())
       .groupBy("customer_id")
       .agg(F.count("*").alias("orders"),
            F.min_by("order_id", by_time).alias("first_order"),
            F.max_by("order_id", by_time).alias("last_order"),
            F.max_by("status", by_time).alias("last_status"))
       .orderBy("customer_id")
       .show())

+-----------+------+-----------+----------+-----------+
|customer_id|orders|first_order|last_order|last_status|
+-----------+------+-----------+----------+-----------+
|          1|     4|       1001|      1015|    shipped|
|          2|     3|       1003|      1013|  delivered|
|          3|     3|       1004|      1016|  delivered|
|          4|     2|       1006|      1010|  cancelled|
|          5|     2|       1008|      1014|  delivered|
|          6|     1|       1012|      1012|    shipped|
+-----------+------+-----------+----------+-----------+



### D. The latest version of every record from several snapshots
Two (or more) copies of a table taken at different times, and you need the current state, including records that exist in only one copy:
1. Tag each copy with its date: `withColumn("snapshot", F.lit(...))`.
2. `unionByName` them.
3. `row_number` over `partitionBy(key).orderBy(F.desc("snapshot"))`, then keep 1.

The same recipe handles a single table with repeated keys and an "updated_at" column (Zillacode #30 removes duplicated keys this way).

In [ ]:
prices_jan = spark.createDataFrame([(101, 1250.0), (102, 150.0), (103, 90.0)], "product_id int, price double")
prices_feb = spark.createDataFrame([(101, 1200.0), (103, 85.0), (104, 60.0)], "product_id int, price double")
history = (prices_jan.withColumn("as_of", F.to_date(F.lit("2024-01-31")))
                     .unionByName(prices_feb.withColumn("as_of", F.to_date(F.lit("2024-02-29")))))
w = W.partitionBy("product_id").orderBy(F.desc("as_of"))
history.withColumn("rn", F.row_number().over(w)).filter(F.col("rn") == 1).drop("rn").orderBy("product_id").show()

+----------+------+----------+
|product_id| price|     as_of|
+----------+------+----------+
|       101|1200.0|2024-02-29|
|       102| 150.0|2024-01-31|
|       103|  85.0|2024-02-29|
|       104|  60.0|2024-02-29|
+----------+------+----------+



### E. Comparing every row with the whole table: three ways
"Products more expensive than the average", "rides whose rating is far from the overall average" (#32). The overall value can come from:
1. **A Python value**: `avg = df.agg(F.avg("x")).first()[0]`, then `F.lit(avg)`. Short, but it runs a separate job.
2. **A cross join with a one-row aggregate**: `df.crossJoin(df.agg(F.avg("x").alias("avg_x")))`. Stays in one plan.
3. **A window over the whole table**: `F.avg("x").over(W.partitionBy())`. No join, but every row goes to one partition.

All three give the same answer; on Zillacode's small inputs, choose whichever reads best.

In [ ]:
avg_price = products.agg(F.avg("price")).first()[0]
one = products.select("product_name", "price", F.lit(avg_price).alias("avg_all"))
two = products.crossJoin(products.agg(F.avg("price").alias("avg_all"))).select("product_name", "price", "avg_all")
three = products.select("product_name", "price", F.avg("price").over(W.partitionBy()).alias("avg_all"))
three.orderBy("product_name").show()
print("same result all three ways:", sorted(one.collect()) == sorted(two.collect()) == sorted(three.collect()))

+------------+------+-------+
|product_name| price|avg_all|
+------------+------+-------+
|     Blender|  60.0|  225.0|
|Coffee Maker|  85.0|  225.0|
|    Cookbook|  25.0|  225.0|
|   Desk Lamp|  40.0|  225.0|
|  Headphones| 150.0|  225.0|
|      Laptop|1200.0|  225.0|
|       Novel|  15.0|  225.0|
+------------+------+-------+

same result all three ways: True


### Exercises

**Q1 · `etl(events)`** Split the events (duplicates removed) of customers 1 and 2 into sessions: a new session starts when more than 30 minutes passed since the customer's previous event. Show customer_id, event_time, page and session (1, 2, ... per customer), sorted by customer_id, then event_time.

Expected output:

```
+-----------+-------------------+-------+-------+
|customer_id|         event_time|   page|session|
+-----------+-------------------+-------+-------+
|          1|2024-03-01 09:00:00|   home|      1|
|          1|2024-03-01 09:10:00|product|      1|
|          1|2024-03-01 11:00:00|   home|      2|
|          1|2024-03-02 10:00:00|   cart|      3|
|          1|2024-03-03 10:30:00|   home|      4|
|          1|2024-03-05 08:00:00|   home|      5|
|          2|2024-03-01 20:00:00|   home|      1|
|          2|2024-03-01 20:20:00|product|      1|
|          2|2024-03-01 20:45:00|   cart|      1|
|          2|2024-03-04 18:00:00|   home|      2|
|          2|2024-03-05 18:00:00|   home|      3|
+-----------+-------------------+-------+-------+
```

In [ ]:
def etl(events):
    # Write code here: return a DataFrame
    return events

etl(events).show()

**Q2 · `etl(events)`** What is each customer's longest streak of consecutive days with at least one event? Show customer_id and longest_streak, sorted by customer_id.

Expected output:

```
+-----------+--------------+
|customer_id|longest_streak|
+-----------+--------------+
|          1|             3|
|          2|             2|
|          3|             3|
+-----------+--------------+
```

In [ ]:
def etl(events):
    # Write code here: return a DataFrame
    return events

etl(events).show()

**Q3 · `etl(customers, customers_new)`** Build the current customer list from two snapshots: `customers` (taken on 2024-03-31) and `customers_new` (taken on 2024-04-07). Keep the newest row of every customer, including customers found only in the older snapshot. Show customer_id, name, city and snapshot (the date of that row's snapshot), sorted by customer_id.

Expected output:

```
+-----------+----------------+--------+----------+
|customer_id|            name|    city|  snapshot|
+-----------+----------------+--------+----------+
|          1|    Aziz Karimov|Tashkent|2024-04-07|
|          2|   Bella Schmidt|  Munich|2024-04-07|
|          3|     Carlos Ruiz|  Madrid|2024-04-07|
|          4|Dilnoza Rahimova|Tashkent|2024-04-07|
|          5|     Emma Wilson|  London|2024-04-07|
|          6|    Farid Aliyev|    NULL|2024-04-07|
|          7|       Grace Lee|  Berlin|2024-03-31|
|          8|       Hana Sato|   Osaka|2024-04-07|
+-----------+----------------+--------+----------+
```

In [ ]:
def etl(customers, customers_new):
    # Write code here: return a DataFrame
    return customers

etl(customers, customers_new).show()

## Lesson 2 · Spark SQL inside `etl`

Zillacode's template creates `spark`, so you can answer in SQL whenever that's easier for you:
1. Register each input DataFrame as a temporary view: `orders.createOrReplaceTempView("orders")`.
2. Return `spark.sql("SELECT ...")`, which is a DataFrame like any other.

Everything from Days 12 to 14 has a SQL form: `CASE WHEN ... THEN ... ELSE ... END`, `COALESCE`, `GROUP BY ... HAVING`, every join type (`LEFT ANTI JOIN`, `LEFT SEMI JOIN`, `CROSS JOIN`), window functions (`ROW_NUMBER() OVER (PARTITION BY ... ORDER BY ...)`), `date_format`, `datediff`... You can also mix the two styles: `F.expr("...")` puts a piece of SQL inside DataFrame code.

Two habits: alias every computed column (`AS revenue`), and filter a window result in an outer query (a `WHERE` can't see a window column of the same `SELECT`). [SQL reference](https://spark.apache.org/docs/latest/sql-ref-syntax.html).

🔬 **In the plan.** `spark.sql(...)` and DataFrame code go through the same Catalyst optimizer and produce the **same physical plan** (next cell). Choose the style that reads best for each problem; it never changes the speed. The same goes for `F.expr("...")` and `selectExpr`.

In [ ]:
def etl(orders, products):
    orders.createOrReplaceTempView("orders")
    products.createOrReplaceTempView("products")
    return spark.sql("""
        SELECT p.category, SUM(o.quantity * p.price) AS revenue
        FROM orders o
        JOIN products p ON o.product_id = p.product_id
        WHERE o.status = 'delivered'
        GROUP BY p.category
        ORDER BY revenue DESC
    """)

etl(orders, products).show()

+-----------+-------+
|   category|revenue|
+-----------+-------+
|Electronics| 2850.0|
|      Books|  205.0|
|    Kitchen|   60.0|
+-----------+-------+



In [ ]:
# A window function in SQL, filtered in an outer query
products.createOrReplaceTempView("products")
spark.sql("""
    SELECT category, product_name, price, price_rank
    FROM (SELECT category, product_name, price,
                 RANK() OVER (PARTITION BY category ORDER BY price DESC) AS price_rank
          FROM products)
    WHERE price_rank = 1
    ORDER BY category
""").show()

+-----------+------------+------+----------+
|   category|product_name| price|price_rank|
+-----------+------------+------+----------+
|      Books|    Cookbook|  25.0|         1|
|Electronics|      Laptop|1200.0|         1|
|       Home|   Desk Lamp|  40.0|         1|
|    Kitchen|Coffee Maker|  85.0|         1|
+-----------+------------+------+----------+



In [ ]:
# F.expr: a piece of SQL inside DataFrame code
orders.select(
    "order_id",
    F.expr("CASE WHEN quantity >= 3 THEN 'big' ELSE 'small' END").alias("size"),
    F.expr("date_format(order_date, 'MMM')").alias("month"),
).show(3)

+--------+-----+-----+
|order_id| size|month|
+--------+-----+-----+
|    1001|small|  Jan|
|    1002|  big|  Jan|
|    1003|small|  Jan|
+--------+-----+-----+
only showing top 3 rows


In [ ]:
# The same query in SQL and with DataFrame methods: the physical plans are the same
orders.createOrReplaceTempView("orders")
products.createOrReplaceTempView("products")
spark.sql("""
    SELECT p.category, SUM(o.quantity) AS items
    FROM orders o JOIN products p ON o.product_id = p.product_id
    WHERE o.status = 'delivered'
    GROUP BY p.category
""").explain()
(orders.filter(F.col("status") == "delivered")
       .join(products, "product_id")
       .groupBy("category").agg(F.sum("quantity").alias("items"))
       .explain())

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[category#53], functions=[sum(quantity#58)])
   +- Exchange hashpartitioning(category#53, 200), ENSURE_REQUIREMENTS, [plan_id=1750]
      +- HashAggregate(keys=[category#53], functions=[partial_sum(quantity#58)])
         +- Project [quantity#58, category#53]
            +- BroadcastHashJoin [product_id#57], [product_id#51], Inner, BuildLeft, false, false
               :- BroadcastExchange HashedRelationBroadcastMode(List(cast(input[0, int, true] as bigint)),false), [plan_id=1745]
               :  +- Project [product_id#57, quantity#58]
               :     +- Filter ((isnotnull(status#60) AND (status#60 = delivered)) AND isnotnull(product_id#57))
               :        +- FileScan parquet [product_id#57,quantity#58,status#60] Batched: true, DataFilters: [isnotnull(status#60), (status#60 = delivered), isnotnull(product_id#57)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/Users/you/spark-lab

### Exercise

**Q4 · `etl(products)`** Solve it with `spark.sql`: the cheapest product of each category. Show category, product_name and price, sorted by category.

Expected output:

```
+-----------+------------+-----+
|   category|product_name|price|
+-----------+------------+-----+
|      Books|       Novel| 15.0|
|Electronics|  Headphones|150.0|
|       Home|   Desk Lamp| 40.0|
|    Kitchen|     Blender| 60.0|
+-----------+------------+-----+
```

In [ ]:
def etl(products):
    # Write code here: return a DataFrame
    return products

etl(products).show()

## Lesson 3 · 🔬 Read less: columns, row groups, folders and files

The cheapest data is the data Spark never reads. Four levers (theory Day 4), measured on `sales` with the `read` number of `measure` (the bytes read from files):

**1. Columns.** Parquet stores each column separately, so a query that uses 2 of the 7 columns reads only those two (column pruning, Day 13). `select("*")` before a write, or a cache of the whole table, reads every column.

**2. Row groups, when the data is clustered.** A filter pushed into the scan (Day 13, Lesson 2) skips the row groups whose min/max can't match. But in `sales` the dates are random: every row group holds dates from the whole two years, so a one-month filter can't skip anything. Written **sorted by date**, each file covers a narrow date range, and the same filter skips almost all of them. Sort the data you write by the column you filter on most (`orderBy`, or `repartitionByRange` plus `sortWithinPartitions`).

**3. Partition folders.** `df.write.partitionBy("sale_month")` writes one folder per value: `sale_month=2024-12/`. A filter on that column becomes `PartitionFilters` in the plan, and Spark lists and opens only the matching folders. The column disappears from the files and comes back from the folder names ([partition discovery](https://spark.apache.org/docs/latest/sql-data-sources-parquet.html#partition-discovery)). Partition by a column you filter on often that has **few values** (a month, a country). A column with millions of values (customer_id) makes millions of tiny folders and files.

**4. File size.** Every file costs a task fixed work: list it, open it, read its footer. A read task takes up to `spark.sql.files.maxPartitionBytes` (128 MB) of files, counting `spark.sql.files.openCostInBytes` (4 MB) for opening each one, so thousands of small files mean many more tasks and much more overhead before any data is read ([file source options](https://spark.apache.org/docs/latest/sql-data-sources-generic-options.html)). Aim for files of roughly 100 MB to 1 GB, and choose the number of output files before writing:

| Method | Plan | Shuffle? | Use it to |
|---|---|---|---|
| `df.coalesce(n)` | `Coalesce n` | no | merge partitions into fewer, bigger ones (fewer output files). It can't add partitions, and the work before it runs in only n tasks |
| `df.repartition(n)` | `Exchange RoundRobinPartitioning(n)` | yes | spread the rows evenly over n partitions |
| `df.repartition(n, "col")` | `Exchange hashpartitioning(col, n)` | yes | put all rows of a key in one partition (and one file) |
| `df.repartitionByRange(n, "col")` | `Exchange rangepartitioning(col, n)` | yes | sorted ranges, e.g. files that each hold a range of dates |

In [ ]:
# 1. Columns: 2 of 7 columns, then all of them
measure(sales.select("sale_id", "customer_id"), "2 columns")
measure(sales, "all 7 columns")

2 columns: 0.25 s | 1 jobs, 1 stages, 4 tasks | read 72.5 MB | shuffle 0 B | spill 0 B
all 7 columns: 0.46 s | 1 jobs, 1 stages, 4 tasks | read 141.9 MB | shuffle 0 B | spill 0 B


In [ ]:
# 2. Row groups: the same December filter on files in random date order, then on files sorted by date
import os
if not os.path.exists("data/big/sales_by_date/_SUCCESS"):
    sales.orderBy("sale_date").write.mode("overwrite").parquet("data/big/sales_by_date")
sales_by_date = spark.read.parquet("data/big/sales_by_date")
december = lambda df: df.filter(F.col("sale_date").between("2024-12-01", "2024-12-31")).agg(F.sum("quantity"))
measure(december(sales), "December, random order")
measure(december(sales_by_date), "December, sorted by date")

December, random order: 0.12 s | 2 jobs, 2 stages, 5 tasks | read 17.6 MB | shuffle 236 B | spill 0 B
December, sorted by date: 0.05 s | 2 jobs, 2 stages, 6 tasks | read 371.5 KB | shuffle 287 B | spill 0 B


In [ ]:
# 3. Partition folders: one folder per month, and a filter on the folder column
if not os.path.exists("data/big/sales_by_month/_SUCCESS"):
    (sales.withColumn("sale_month", F.date_format("sale_date", "yyyy-MM"))
          .write.mode("overwrite").partitionBy("sale_month").parquet("data/big/sales_by_month"))
folders = sorted(d for d in os.listdir("data/big/sales_by_month") if d.startswith("sale_month="))
print(len(folders), "folders:", folders[:3], "...")
by_month = spark.read.parquet("data/big/sales_by_month")
dec_folder = by_month.filter(F.col("sale_month") == "2024-12").agg(F.sum("quantity"))
dec_folder.explain()
measure(dec_folder, "December folder only")

24 folders: ['sale_month=2023-01', 'sale_month=2023-02', 'sale_month=2023-03'] ...
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[], functions=[sum(quantity#697)])
   +- Exchange SinglePartition, ENSURE_REQUIREMENTS, [plan_id=2056]
      +- HashAggregate(keys=[], functions=[partial_sum(quantity#697)])
         +- Project [quantity#697]
            +- FileScan parquet [quantity#697,sale_month#700] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/Users/you/spark-lab/data/big/sales_by_month], PartitionFilters: [isnotnull(sale_month#700), (sale_month#700 = 2024-12)], PushedFilters: [], ReadSchema: struct<quantity:int>


December folder only: 0.09 s | 2 jobs, 2 stages, 5 tasks | read 290.7 KB | shuffle 236 B | spill 0 B


In [ ]:
# 4. Small files: the same table as 500 files
if not os.path.exists("data/big/sales_500_files/_SUCCESS"):
    sales.repartition(500).write.mode("overwrite").parquet("data/big/sales_500_files")
many_files = spark.read.parquet("data/big/sales_500_files")
print("read partitions:", many_files.rdd.getNumPartitions(), "for 500 files vs", sales.rdd.getNumPartitions(), "for 8")
measure(many_files.agg(F.sum("quantity")), "sum, 500 files")
measure(sales.agg(F.sum("quantity")), "sum, 8 files")

read partitions: 17 for 500 files vs 4 for 8
sum, 500 files: 0.21 s | 2 jobs, 2 stages, 18 tasks | read 23.8 MB | shuffle 1.0 KB | spill 0 B
sum, 8 files: 0.04 s | 2 jobs, 2 stages, 5 tasks | read 4.3 MB | shuffle 236 B | spill 0 B


In [ ]:
# coalesce merges partitions without a shuffle; repartition shuffles
sales.coalesce(2).explain()
sales.repartition(8, "store_id").explain()

== Physical Plan ==
Coalesce 2
+- *(1) ColumnarToRow
   +- FileScan parquet [sale_id#109L,customer_id#110,product_id#111,store_id#112,quantity#113,sale_date#114,channel#115] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/Users/you/spark-lab/data/big/sales], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<sale_id:bigint,customer_id:int,product_id:int,store_id:int,quantity:int,sale_date:date,channel:string>


== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Exchange hashpartitioning(store_id#112, 8), REPARTITION_BY_NUM, [plan_id=2268]
   +- FileScan parquet [sale_id#109L,customer_id#110,product_id#111,store_id#112,quantity#113,sale_date#114,channel#115] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/Users/you/spark-lab/data/big/sales], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<sale_id:bigint,customer_id:int,product_id:int,store_id:int,quantity:int,sale_date:da

### Exercises

**Q5 · 🔍 Read the plan** **Predict** which of the three queries reads the fewest bytes and which the most, then run the cell. Why does (b) read so much less than (a), although its plan has no `PartitionFilters`?

In [ ]:
dec_dates = lambda df: df.filter(F.col("sale_date").between("2024-12-01", "2024-12-31")).agg(F.sum("quantity"))
measure(dec_dates(sales), "(a) sales, filter on sale_date")
measure(dec_dates(by_month), "(b) month folders, filter on sale_date")
measure(by_month.filter(F.col("sale_month") == "2024-12").agg(F.sum("quantity")), "(c) month folders, filter on sale_month")
dec_dates(by_month).explain()
# Your answer:

**Q6 · ⚡ `etl(by_month)`** Add up the items of each channel in the first quarter of 2024 (January to March), sorted by channel. `slow_etl` gets it right but reads every month folder. Make Spark open only three of them.

Expected output:

```
+-------+-------+
|channel|  items|
+-------+-------+
|    app|1246124|
|  store|1243699|
|    web|1245358|
+-------+-------+
```

**Target:** the plan's `PartitionFilters` select the three months, and `read` drops from about 14 MB to about 1 MB.

In [ ]:
def slow_etl(by_month):
    return (by_month.filter((F.year("sale_date") == 2024) & (F.quarter("sale_date") == 1))
                    .groupBy("channel").agg(F.sum("quantity").alias("items"))
                    .orderBy("channel"))

def etl(by_month):
    # Write code here: return a DataFrame
    return slow_etl(by_month)

measure(slow_etl(by_month), "slow")
measure(etl(by_month), "yours")
etl(by_month).show()
etl(by_month).explain()

## Lesson 4 · 🔬 Caching: compute once, use several times

A DataFrame is a recipe, not data: every action runs the whole recipe again from the files. Three reports built on one expensive DataFrame compute it three times. `df.cache()` keeps the computed rows the first time an action needs them, and later actions read them from there (theory Day 7; [caching in the tuning guide](https://spark.apache.org/docs/latest/sql-performance-tuning.html#caching-data)).

* `cache()` is **lazy**: it only marks the DataFrame. The first action fills the cache, and it's the slowest one: it computes everything *and* stores it.
* For DataFrames, `cache()` is `persist(StorageLevel.MEMORY_AND_DISK)`: in memory, and on disk what doesn't fit. `persist(StorageLevel.DISK_ONLY)` and the other levels trade memory for speed ([persist docs](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.persist.html)).
* In the plan, the cached part becomes an `InMemoryRelation` read by `InMemoryTableScan`. The Spark UI's **Storage** tab lists every cached DataFrame with its size in memory and on disk.
* `df.unpersist()` frees it (`spark.catalog.clearCache()` frees everything). Until then it stays, unless Spark needs the memory for running tasks.
* A cached DataFrame keeps the partitions it had when it was cached. AQE can't coalesce them afterwards, so a join or aggregate result cached with 200 partitions is read by 200 tasks.

**It pays when** the same DataFrame is used by several actions, it's expensive to compute (a join, a big aggregation, a window), and what you store is small enough to fit in memory: an aggregate is ideal.

**It hurts when** the DataFrame is used once (you pay to store it for nothing), or when you cache a raw table and then filter it: the cache holds every row and column, and the filter can no longer reach the Parquet reader (last cell). And `cache()` inside a loop without `unpersist()` fills the memory.

**Cache, checkpoint or a file?** A cache keeps the lineage: if an executor is lost, Spark recomputes the missing partitions. `df.checkpoint()` writes the data to reliable storage and cuts the lineage, which helps very long plans (loops that build on their own result). Writing an intermediate result to Parquet is the most robust choice: another job can read it tomorrow.

In [ ]:
# An expensive DataFrame (a full pass over 10 million sales, with a shuffle) that comes out small: one row per customer
per_customer = (sales.groupBy("customer_id")
                     .agg(F.count("*").alias("purchases"), F.sum("quantity").alias("items"),
                          F.min("sale_date").alias("first_sale"), F.max("sale_date").alias("last_sale")))
def three_reports():
    per_customer.groupBy("purchases").count().collect()                            # customers by number of purchases
    per_customer.orderBy(F.desc("items")).limit(10).collect()                       # the top 10 customers
    per_customer.groupBy(F.date_format("first_sale", "yyyy-MM")).count().collect()   # new customers per month
measure(three_reports, "3 reports, no cache")
per_customer.cache()
measure(lambda: per_customer.count(), "fill the cache")
measure(three_reports, "3 reports from the cache")
for r in ui("storage/rdd"):                     # what the Storage tab shows
    print("cached:", size(r["memoryUsed"]), "in memory,", size(r["diskUsed"]), "on disk,",
          r["numCachedPartitions"], "of", r["numPartitions"], "partitions")

3 reports, no cache: 4.06 s | 8 jobs, 8 stages, 26 tasks | read 126.5 MB | shuffle 173.4 MB | spill 0 B
fill the cache: 1.76 s | 4 jobs, 4 stages, 405 tasks | read 69.9 MB | shuffle 90.3 MB | spill 0 B
3 reports from the cache: 1.54 s | 5 jobs, 5 stages, 602 tasks | read 62.1 MB | shuffle 470.0 KB | spill 0 B
cached: 20.7 MB in memory, 0 B on disk, 200 of 200 partitions


In [ ]:
# In the plan, the reports read the cache (InMemoryTableScan) instead of the files
per_customer.groupBy("purchases").count().explain()
print("still cached:", per_customer.unpersist().is_cached)      # unpersist() frees the memory

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[purchases#868L], functions=[count(1)])
   +- Exchange hashpartitioning(purchases#868L, 200), ENSURE_REQUIREMENTS, [plan_id=3232]
      +- HashAggregate(keys=[purchases#868L], functions=[partial_count(1)])
         +- InMemoryTableScan [purchases#868L]
               +- InMemoryRelation [customer_id#110, purchases#868L, items#869L, first_sale#870, last_sale#871], StorageLevel(disk, memory, deserialized, 1 replicas)
                     +- AdaptiveSparkPlan isFinalPlan=true
                        +- == Final Plan ==
                           ResultQueryStage 1
                           +- *(2) HashAggregate(keys=[customer_id#110], functions=[count(1), sum(quantity#113), min(sale_date#114), max(sale_date#114)])
                              +- ShuffleQueryStage 0
                                 +- Exchange hashpartitioning(customer_id#110, 200), ENSURE_REQUIREMENTS, [plan_id=3053]
                          

In [ ]:
# Caching the raw table, then filtering: every row is cached, and the filter can't reach the Parquet reader
sales.cache()
sales.filter(F.col("store_id") == 7).explain()
print("still cached:", sales.unpersist().is_cached)

== Physical Plan ==
*(1) Filter (isnotnull(store_id#112) AND (store_id#112 = 7))
+- InMemoryTableScan [sale_id#109L, customer_id#110, product_id#111, store_id#112, quantity#113, sale_date#114, channel#115], [isnotnull(store_id#112), (store_id#112 = 7)]
      +- InMemoryRelation [sale_id#109L, customer_id#110, product_id#111, store_id#112, quantity#113, sale_date#114, channel#115], StorageLevel(disk, memory, deserialized, 1 replicas)
            +- *(1) ColumnarToRow
               +- FileScan parquet [sale_id#109L,customer_id#110,product_id#111,store_id#112,quantity#113,sale_date#114,channel#115] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/Users/you/spark-lab/data/big/sales], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<sale_id:bigint,customer_id:int,product_id:int,store_id:int,quantity:int,sale_date:date,channel:string>


still cached: False


### Exercise

**Q7 · 🔍 Think it through** Which of these four programs gain from `cache()`, and which lose? Answer without running them.

In [ ]:
# (a) big = sales.join(customers_big, "customer_id").cache()
#     print(big.count())
#
# (b) per_store = sales.groupBy("store_id", "channel").agg(F.sum("quantity").alias("items")).cache()
#     then 5 different reports are built from per_store
#
# (c) sales.cache()
#     sales.filter(F.col("store_id") == 7).show()
#
# (d) clean = (an expensive cleaning of sales: joins, windows).cache()
#     for month in the 24 months: clean.filter(month).write.parquet(one folder per month)
#
# Your answers:

## Lesson 5 · UDFs: a Python function as a column function

`F.udf(python_function, return_type)` turns a plain Python function into a column function. Some official Zillacode solutions use one (#31 checks weekends with Python's `datetime`, #40 counts list items with Python's `split`). It works, but:
* A UDF runs Python for every row, outside Spark's optimizer, so it's slower than built-in functions (theory Day 8).
* **The function itself must handle `None`**: Spark passes null as `None`, and `None.split(",")` crashes the job.
* Almost every case has a built-in: `F.dayofweek(F.to_date(col, "MM/dd/yyyy")).isin(1, 7)` for weekends, `F.size(F.split(...))` with a null guard for counting.

Use a UDF only when no built-in function (or combination of them) can do the job. [udf docs](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.udf.html).

🔬 **What a UDF costs.** Spark can't look inside a Python function, so: (1) **every row** is sent from the JVM to a Python worker process and the result comes back; (2) a condition that uses a UDF can't be pushed into the scan (`PushedFilters` stays empty), and the UDF breaks the stage's generated code; (3) Python runs much slower than the JVM code of built-in functions. The plan shows where rows leave the JVM: `BatchEvalPython` (rows pickled in batches, the default in Spark 3 and so on Zillacode) or `ArrowEvalPython` (rows in Arrow column batches: the default for `F.udf` in Spark 4.2, and always for `pandas_udf`). The choice is made when the UDF is created: `F.udf(f, "string", useArrow=False)` forces pickled rows. The next cells measure one rule four ways on 10 million rows. Expect the built-in to win, the pickled UDF to lose, and Arrow and pandas UDFs in between; the first Python UDF of a session also pays for starting the Python workers, hence the warm-up. [Arrow and pandas UDFs](https://spark.apache.org/docs/latest/api/python/tutorial/sql/arrow_pandas.html).

In [ ]:
import datetime                                   # Zillacode's template imports the module like this
from pyspark.sql.types import IntegerType, BooleanType

count_items = F.udf(lambda s: len(s.split(",")) if s else 0, IntegerType())        # handles None and ""
is_weekend = F.udf(lambda s: datetime.datetime.strptime(s, "%m/%d/%Y").weekday() >= 5 if s else None, BooleanType())

t = spark.createDataFrame([("a,b,c", "03/02/2024"), ("", "03/04/2024"), (None, None)], "items string, day string")
t.select(
    "items", "day",
    count_items("items").alias("count_udf"),
    F.when(F.col("items").isNull() | (F.col("items") == ""), 0).otherwise(F.size(F.split("items", ","))).alias("count_builtin"),
    is_weekend("day").alias("weekend_udf"),
    F.dayofweek(F.to_date("day", "MM/dd/yyyy")).isin(1, 7).alias("weekend_builtin"),
).show()

+-----+----------+---------+-------------+-----------+---------------+
|items|       day|count_udf|count_builtin|weekend_udf|weekend_builtin|
+-----+----------+---------+-------------+-----------+---------------+
|a,b,c|03/02/2024|        3|            3|       true|           true|
|     |03/04/2024|        0|            0|      false|          false|
| NULL|      NULL|        0|            0|       NULL|           NULL|
+-----+----------+---------+-------------+-----------+---------------+



In [ ]:
# One rule (big = 4 items or more) four ways. The Arrow choice is fixed when a UDF is created:
# useArrow=True is Spark 4.2's default for F.udf; useArrow=False pickles rows, like Spark 3 (and Zillacode)
import numpy as np, pandas as pd
from pyspark.sql.functions import pandas_udf
band = lambda q: "big" if q >= 4 else "small"
udf_arrow = F.udf(band, "string", useArrow=True)
udf_pickled = F.udf(band, "string", useArrow=False)
@pandas_udf("string")
def band_pandas(q: pd.Series) -> pd.Series:          # gets a whole batch as a pandas Series
    return pd.Series(np.where(q >= 4, "big", "small"))

measure(sales.limit(10_000).select(udf_arrow("quantity"), udf_pickled("quantity"), band_pandas("quantity")),
        "warm-up: starts the Python workers")
measure(sales.select("sale_id", F.when(F.col("quantity") >= 4, "big").otherwise("small").alias("band")), "built-in")
measure(sales.select("sale_id", udf_arrow("quantity").alias("band")), "udf, Arrow")
measure(sales.select("sale_id", udf_pickled("quantity").alias("band")), "udf, pickled")
measure(sales.select("sale_id", band_pandas("quantity").alias("band")), "pandas_udf")

warm-up: starts the Python workers: 0.55 s | 2 jobs, 2 stages, 5 tasks | read 2.1 MB | shuffle 86.9 KB | spill 0 B
built-in: 0.17 s | 1 jobs, 1 stages, 4 tasks | read 45.0 MB | shuffle 0 B | spill 0 B
udf, Arrow: 1.13 s | 1 jobs, 1 stages, 4 tasks | read 45.0 MB | shuffle 0 B | spill 0 B
udf, pickled: 1.41 s | 1 jobs, 1 stages, 4 tasks | read 45.0 MB | shuffle 0 B | spill 0 B
pandas_udf: 0.60 s | 1 jobs, 1 stages, 4 tasks | read 45.0 MB | shuffle 0 B | spill 0 B


In [ ]:
# The plans: ArrowEvalPython, BatchEvalPython, and the built-in condition, which is the only one pushed into the scan
sales.filter(udf_arrow("quantity") == "big").select("sale_id").explain()
sales.filter(udf_pickled("quantity") == "big").select("sale_id").explain()
sales.filter(F.col("quantity") >= 4).select("sale_id").explain()

== Physical Plan ==
*(2) Project [sale_id#109L]
+- *(2) Filter (pythonUDF0#1729 = big)
   +- ArrowEvalPython [<lambda>(quantity#113)#1728], [pythonUDF0#1729], 101
      +- *(1) ColumnarToRow
         +- FileScan parquet [sale_id#109L,quantity#113] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/Users/you/spark-lab/data/big/sales], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<sale_id:bigint,quantity:int>


== Physical Plan ==
*(2) Project [sale_id#109L]
+- *(2) Filter (pythonUDF0#1731 = big)
   +- BatchEvalPython [<lambda>(quantity#113)#1730], [pythonUDF0#1731]
      +- *(1) ColumnarToRow
         +- FileScan parquet [sale_id#109L,quantity#113] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/Users/you/spark-lab/data/big/sales], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<sale_id:bigint,quantity:int>


== Physical Plan ==
*(1) Project [sale_id#109L]
+- *(1) Filter (isnot

### Exercise

**Q8 · ⚡ `etl(sales)`** Count the weekend sales (Saturday or Sunday) of each channel, sorted by channel. `slow_etl` does it with a Python UDF; do it with built-in functions only.

Expected output:

```
+-------+-------------+
|channel|weekend_sales|
+-------+-------------+
|    app|       952034|
|  store|       953758|
|    web|       952922|
+-------+-------------+
```

**Target:** no `ArrowEvalPython` or `BatchEvalPython` in the plan, and several times faster in `measure`.

In [ ]:
def slow_etl(sales):
    is_weekend = F.udf(lambda d: d.weekday() >= 5, "boolean")      # Python: Monday is 0, Sunday is 6
    return (sales.filter(is_weekend("sale_date"))
                 .groupBy("channel").agg(F.count("*").alias("weekend_sales"))
                 .orderBy("channel"))

def etl(sales):
    # Write code here: return a DataFrame
    return slow_etl(sales)

measure(slow_etl(sales), "slow")
measure(etl(sales), "yours")
etl(sales).show()
etl(sales).explain()

## Lesson 6 · Debugging: errors, and the mistakes that give no error

**The errors you will meet**, and what they mean:

| Message (start) | Cause | Fix |
|---|---|---|
| `[UNRESOLVED_COLUMN.WITH_SUGGESTION] ... cannot be resolved. Did you mean ...` | a typo, or a column that an earlier `select`/`groupBy` removed | read the suggested names; `print(df.columns)` |
| `[AMBIGUOUS_REFERENCE]` | both sides of a join have that column | Day 14, Lesson 3 |
| `[CANNOT_CONVERT_COLUMN_INTO_BOOL]` | `and` / `or` / `not` / `if` used on a column | `&`, `\|`, `~`, `F.when` |
| `Column is not iterable` | Python's own `sum`, `max`, `len` used on a column | `F.sum`, `F.max`, `F.length` |
| `'Column' object is not callable` | a DataFrame method called on a column (`F.col("x").count()`) | `F.count("x")` inside `agg` |
| `'NoneType' object has no attribute ...`, or Zillacode's *etl function must return ... not NoneType* | the function returns nothing, or returns `df.show()` (which prints and returns `None`) | `return df` |
| `[DATATYPE_MISMATCH]` | e.g. `F.size` on a text, or `union` of incompatible columns | check the types with `df.printSchema()` |

**Mistakes that give no error, just a wrong answer.** Check these before you press Run:
1. **Columns**: exactly the output schema's names; no helper columns (`rn`, `gap`, `flag`) left over.
2. **Types**: an `avg` or a `/` gives a double; if the schema says Integer, `cast("int")`. Round only when the task says so, and to the stated number of decimals.
3. **Nulls**: filters with `!=` drop null rows; `count("*")` vs `count(col)`; left joins need `coalesce(..., 0)` when the task wants 0.
4. **Joins**: "all X" means a left join; two detail tables joined before aggregating multiply each other (the fan-out trap); null keys never match.
5. **Ties**: `row_number` (exactly N rows) vs `rank` (gaps) vs `dense_rank` (no gaps), and a tie-breaker where the task names one.
6. **Duplicates**: `distinct` before counting days or pages; `count_distinct` when the task says "different" or "unique".
7. **Dates**: text input means `to_date`/`to_timestamp` with the right pattern (`MM` months, `mm` minutes).
8. **Hidden test**: no values copied from the example (ids, dates, counts).

### 🔬 Performance mistakes that give no error

| Mistake | Why it's slow | Instead |
|---|---|---|
| `df.collect()` or `df.toPandas()` on a big table | every row travels to the driver, one process on one machine; it can fail with an out-of-memory error or `spark.driver.maxResultSize` | aggregate or `limit` first; write big results to files |
| `if df.count() > 0:` | counts every row to answer yes or no | `if not df.isEmpty():` stops at the first row |
| `df = df.withColumn(...)` in a loop over hundreds of columns | each call analyzes a bigger plan on the driver, before any data is read | one `select(...)` or `withColumns({...})` |
| the same expensive DataFrame in several actions, without a cache | each action computes it again | `cache()` it (Lesson 4) |
| `for row in df.collect():` with Spark calls inside the loop | one Spark job per row | one join or `groupBy` |
| `show()` to check that a query is fast | `show()` runs only enough of the plan to print 20 rows (`CollectLimit`) | `measure()`, or a write |
| `coalesce(1)` / `repartition(1)` before writing a big table | the whole table goes through one task (and `coalesce(1)` pulls the work before it into that task too) | let Spark write several files of a sensible size |
| an `orderBy` nobody needs | a full shuffle and sort (Day 13, Q1) | sort only the final, small result |

In [ ]:
# A typo: the message suggests the real names
try:
    orders.select("order_dte")
except Exception as e:
    print(type(e).__name__, "->", str(e).split("SQLSTATE")[0][:230])

AnalysisException -> [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `order_dte` cannot be resolved. Did you mean one of the following? [`order_date`, `order_id`, `product_id`, `quantity`, `status`]. 


In [ ]:
# show() prints and returns None: never assign or return it
result = orders.filter(F.col("quantity") > 3).show()
print("result is", result)

+--------+-----------+----------+--------+----------+---------+
|order_id|customer_id|product_id|quantity|order_date|   status|
+--------+-----------+----------+--------+----------+---------+
|    1009|          3|       105|       4|2024-03-05|delivered|
|    1014|          5|       105|       5|2024-03-20|delivered|
+--------+-----------+----------+--------+----------+---------+

result is None


In [ ]:
# Python's own functions on a column, and a DataFrame method on a column
for attempt in [lambda: sum(F.col("quantity")), lambda: F.col("quantity").count()]:
    try:
        attempt()
    except Exception as e:
        print(type(e).__name__, "->", str(e).split(".")[0][:120])

PySparkTypeError -> [NOT_ITERABLE] Column is not iterable
TypeError -> 'Column' object is not callable


In [ ]:
# Is there a big order? count() reads every row; isEmpty() stops at the first one
big_orders = sales.filter(F.col("quantity") == 5)
measure(lambda: big_orders.count() > 0, "count() > 0")
measure(lambda: big_orders.isEmpty(), "isEmpty()")

count() > 0: 0.08 s | 2 jobs, 2 stages, 5 tasks | read 5.0 MB | shuffle 236 B | spill 0 B
isEmpty(): 0.03 s | 1 jobs, 2 stages, 5 tasks | read 2.5 MB | shuffle 184 B | spill 0 B


In [ ]:
# 200 new columns: a withColumn loop vs one withColumns call. Only the plan is built here, nothing runs
start = time.perf_counter()
df = sales
for i in range(200):
    df = df.withColumn(f"c{i}", F.col("quantity") + i)
print(f"200 x withColumn: {time.perf_counter() - start:.2f} s")
start = time.perf_counter()
df = sales.withColumns({f"c{i}": F.col("quantity") + i for i in range(200)})
print(f"1 x withColumns:  {time.perf_counter() - start:.2f} s")

200 x withColumn: 1.38 s
1 x withColumns:  0.21 s


### Exercise

**Q9 · 🔍 Read the code** This function works on a small table and would hurt on `sales`. Find four problems, without running it.

In [ ]:
# def daily_report(sales, customers_big):
#     if sales.count() == 0:
#         return {}
#     report = sales.join(customers_big, "customer_id").orderBy("sale_date")
#     for channel in ["web", "app", "store"]:
#         report = report.withColumn(f"is_{channel}", F.when(F.col("channel") == channel, 1).otherwise(0))
#     totals = {}
#     for row in report.collect():
#         totals[row["city"]] = totals.get(row["city"], 0) + row["quantity"]
#     return totals
#
# Your answer:

## The slow report: a final optimization problem

A real job, as it often looks before anyone reads its plan. It returns the right numbers, but it makes four of the mistakes you've met in Days 13 to 15. Find them in `explain()` and `measure`, then fix them.

**Q10 · ⚡ `etl(sales, customers_big)`** For 2024, show for each city: sales (the number of sales), items, customers (the number of different customers) and weekend_pct (the percentage of the city's sales made on a Saturday or Sunday, 1 decimal), sorted by city. `slow_etl` returns exactly this. Write a version without its four problems.

Expected output:

```
+---------+-------+-------+---------+-----------+
|     city|  sales|  items|customers|weekend_pct|
+---------+-------+-------+---------+-----------+
|   Berlin| 350551|1050390|   165371|       28.4|
|  Bukhara| 351129|1054126|   165230|       28.4|
|   London| 350391|1052052|   165255|       28.4|
|   Madrid| 350491|1051790|   165527|       28.5|
|   Munich| 350579|1051998|   165400|       28.4|
|    Osaka| 350956|1053709|   165289|       28.4|
|    Paris| 350578|1050359|   165432|       28.4|
|     Rome| 351569|1054367|   165730|       28.5|
|Samarkand|1850566|5550118|   165340|       28.4|
| Tashkent| 351930|1056961|   165424|       28.5|
+---------+-------+-------+---------+-----------+
```

**Target:** no `ArrowEvalPython`/`BatchEvalPython`, no `InMemoryTableScan`, a `BroadcastHashJoin`, and the date filter in `PushedFilters`. `measure` should be well under half the slow time. (The cell clears Spark's cache first, so the slow version is measured from scratch every time.)

In [ ]:
def slow_etl(sales, customers_big):
    is_weekend = F.udf(lambda d: d.weekday() >= 5, "boolean")
    in_2024 = sales.filter(F.year("sale_date") == 2024)
    joined = in_2024.join(customers_big, "customer_id").cache()
    per_city = joined.groupBy("city").agg(F.count("*").alias("sales"), F.sum("quantity").alias("items"))
    buyers = joined.groupBy("city").agg(F.countDistinct("customer_id").alias("customers"))
    weekend = joined.filter(is_weekend("sale_date")).groupBy("city").agg(F.count("*").alias("weekend_sales"))
    return (per_city.join(buyers, "city").join(weekend, "city")
                    .withColumn("weekend_pct", F.round(F.col("weekend_sales") / F.col("sales") * 100, 1))
                    .select("city", "sales", "items", "customers", "weekend_pct")
                    .orderBy("city"))

def etl(sales, customers_big):
    # Write code here: return a DataFrame
    return slow_etl(sales, customers_big)

spark.catalog.clearCache()                       # measure the slow version from scratch
measure(slow_etl(sales, customers_big), "slow")
spark.catalog.clearCache()
measure(etl(sales, customers_big), "yours")
etl(sales, customers_big).show()
etl(sales, customers_big).explain()
spark.catalog.clearCache()

## Mock exam: 6 problems in 75 minutes

Set a timer for 75 minutes. No hints and no looking back at the lessons: write each `etl` from scratch, as on Zillacode. Suggested time: easy 5 minutes, medium 10, hard 15; what's left is your buffer. Check your answers only when the time is up.

**New data, built like Zillacode builds it.** A bike-sharing service in Uzbekistan. The cell below creates the tables from lists of Python dicts with `spark.createDataFrame`, exactly as Zillacode does, so:
* the columns are in **alphabetical order**,
* whole numbers are `bigint`, and decimals are `double`,
* **times and dates are text**.

| Table | Columns |
|---|---|
| **stations** | *station_id*, *station_name*, *city* |
| **members** | *member_id*, *full_name*, *plan* (`annual`, `monthly` or `trial`), *joined* (text `MM/dd/yyyy`, or `unknown`) |
| **rides** | *ride_id*, *member_id*, *bike_id*, *start_station*, *end_station* (station ids), *start_time*, *end_time* (text `yyyy-MM-dd HH:mm:ss`), *price* (dollars) |

In [ ]:
stations = spark.createDataFrame([
    {"station_id": 1, "station_name": "Amir Temur Square", "city": "Tashkent"},
    {"station_id": 2, "station_name": "Chorsu Bazaar", "city": "Tashkent"},
    {"station_id": 3, "station_name": "Registan", "city": "Samarkand"},
    {"station_id": 4, "station_name": "Siab Bazaar", "city": "Samarkand"},
    {"station_id": 5, "station_name": "Lyabi-Hauz", "city": "Bukhara"},
])
members = spark.createDataFrame([
    {"member_id": 1, "full_name": "Nodira Yusupova", "plan": "annual", "joined": "01/15/2024"},
    {"member_id": 2, "full_name": "Timur Saidov", "plan": "monthly", "joined": "02/03/2024"},
    {"member_id": 3, "full_name": "Laylo Karimova", "plan": "annual", "joined": "02/17/2024"},
    {"member_id": 4, "full_name": "Javlon Rashidov", "plan": "monthly", "joined": "03/02/2024"},
    {"member_id": 5, "full_name": "Malika Tursunova", "plan": "trial", "joined": "unknown"},
])
rides = spark.createDataFrame([
    {"ride_id": 1, "member_id": 1, "bike_id": "B1", "start_station": 1, "end_station": 2, "start_time": "2024-03-01 08:00:00", "end_time": "2024-03-01 08:25:00", "price": 1.5},
    {"ride_id": 2, "member_id": 2, "bike_id": "B2", "start_station": 3, "end_station": 4, "start_time": "2024-03-01 09:00:00", "end_time": "2024-03-01 09:20:00", "price": 1.2},
    {"ride_id": 3, "member_id": 1, "bike_id": "B1", "start_station": 2, "end_station": 1, "start_time": "2024-03-01 18:10:00", "end_time": "2024-03-01 18:50:00", "price": 2.4},
    {"ride_id": 4, "member_id": 1, "bike_id": "B1", "start_station": 1, "end_station": 1, "start_time": "2024-03-02 07:45:00", "end_time": "2024-03-02 08:30:00", "price": 2.7},
    {"ride_id": 5, "member_id": 3, "bike_id": "B2", "start_station": 3, "end_station": 3, "start_time": "2024-03-02 10:00:00", "end_time": "2024-03-02 10:35:00", "price": 2.1},
    {"ride_id": 6, "member_id": 2, "bike_id": "B3", "start_station": 5, "end_station": 5, "start_time": "2024-03-02 16:00:00", "end_time": "2024-03-02 16:15:00", "price": 0.9},
    {"ride_id": 7, "member_id": 1, "bike_id": "B1", "start_station": 2, "end_station": 2, "start_time": "2024-03-03 08:05:00", "end_time": "2024-03-03 08:20:00", "price": 0.9},
    {"ride_id": 8, "member_id": 3, "bike_id": "B2", "start_station": 3, "end_station": 4, "start_time": "2024-03-03 11:00:00", "end_time": "2024-03-03 11:50:00", "price": 3.0},
    {"ride_id": 9, "member_id": 2, "bike_id": "B3", "start_station": 5, "end_station": 6, "start_time": "2024-03-04 12:00:00", "end_time": "2024-03-04 12:30:00", "price": 1.8},
    {"ride_id": 10, "member_id": 3, "bike_id": "B1", "start_station": 2, "end_station": 1, "start_time": "2024-03-04 19:00:00", "end_time": "2024-03-04 19:10:00", "price": 0.6},
    {"ride_id": 11, "member_id": 1, "bike_id": "B2", "start_station": 4, "end_station": 3, "start_time": "2024-03-05 09:00:00", "end_time": "2024-03-05 09:40:00", "price": 2.4},
    {"ride_id": 12, "member_id": 4, "bike_id": "B3", "start_station": 1, "end_station": 2, "start_time": "2024-03-06 08:00:00", "end_time": "2024-03-06 08:35:00", "price": 2.1},
    {"ride_id": 13, "member_id": 3, "bike_id": "B3", "start_station": 1, "end_station": 1, "start_time": "2024-03-05 17:00:00", "end_time": "2024-03-05 17:20:00", "price": 1.2},
])
rides.printSchema()
stations.show()
members.show()
rides.show(truncate=False)

root
 |-- bike_id: string (nullable = true)
 |-- end_station: long (nullable = true)
 |-- end_time: string (nullable = true)
 |-- member_id: long (nullable = true)
 |-- price: double (nullable = true)
 |-- ride_id: long (nullable = true)
 |-- start_station: long (nullable = true)
 |-- start_time: string (nullable = true)

+---------+----------+-----------------+
|     city|station_id|     station_name|
+---------+----------+-----------------+
| Tashkent|         1|Amir Temur Square|
| Tashkent|         2|    Chorsu Bazaar|
|Samarkand|         3|         Registan|
|Samarkand|         4|      Siab Bazaar|
|  Bukhara|         5|       Lyabi-Hauz|
+---------+----------+-----------------+

+----------------+----------+---------+-------+
|       full_name|    joined|member_id|   plan|
+----------------+----------+---------+-------+
| Nodira Yusupova|01/15/2024|        1| annual|
|    Timur Saidov|02/03/2024|        2|monthly|
|  Laylo Karimova|02/17/2024|        3| annual|
| Javlon Rashidov|

**Q11 · `etl(rides)`** (easy) Which rides lasted more than 30 minutes? Show ride_id, member_id and minutes (a whole number), sorted by ride_id.

Expected output:

```
+-------+---------+-------+
|ride_id|member_id|minutes|
+-------+---------+-------+
|      3|        1|     40|
|      4|        1|     45|
|      5|        3|     35|
|      8|        3|     50|
|     11|        1|     40|
|     12|        4|     35|
+-------+---------+-------+
```

In [ ]:
def etl(rides):
    # Write code here: return a DataFrame
    return rides

etl(rides).show()

**Q12 · `etl(members, rides)`** (easy) Which members have never taken a ride? Show member_id and full_name.

Expected output:

```
+---------+----------------+
|member_id|       full_name|
+---------+----------------+
|        5|Malika Tursunova|
+---------+----------------+
```

In [ ]:
def etl(members, rides):
    # Write code here: return a DataFrame
    return members

etl(members, rides).show()

**Q13 · `etl(rides, stations)`** (medium) Show every ride with the names of its stations: ride_id, from_station and to_station (empty when the station is unknown), sorted by ride_id.

Expected output:

```
+-------+-----------------+-----------------+
|ride_id|     from_station|       to_station|
+-------+-----------------+-----------------+
|      1|Amir Temur Square|    Chorsu Bazaar|
|      2|         Registan|      Siab Bazaar|
|      3|    Chorsu Bazaar|Amir Temur Square|
|      4|Amir Temur Square|Amir Temur Square|
|      5|         Registan|         Registan|
|      6|       Lyabi-Hauz|       Lyabi-Hauz|
|      7|    Chorsu Bazaar|    Chorsu Bazaar|
|      8|         Registan|      Siab Bazaar|
|      9|       Lyabi-Hauz|             NULL|
|     10|    Chorsu Bazaar|Amir Temur Square|
|     11|      Siab Bazaar|         Registan|
|     12|Amir Temur Square|    Chorsu Bazaar|
|     13|Amir Temur Square|Amir Temur Square|
+-------+-----------------+-----------------+
```

In [ ]:
def etl(rides, stations):
    # Write code here: return a DataFrame
    return rides

etl(rides, stations).show()

**Q14 · `etl(rides, stations)`** (medium) Find each member's favourite start station: the one they started the most rides from (on a tie, the smaller station_id). Show member_id, station_name and rides, sorted by member_id.

Expected output:

```
+---------+-----------------+-----+
|member_id|     station_name|rides|
+---------+-----------------+-----+
|        1|Amir Temur Square|    2|
|        2|       Lyabi-Hauz|    2|
|        3|         Registan|    2|
|        4|Amir Temur Square|    1|
+---------+-----------------+-----+
```

In [ ]:
def etl(rides, stations):
    # Write code here: return a DataFrame
    return rides

etl(rides, stations).show()

**Q15 · `etl(rides)`** (hard) A bike should start each ride where its previous ride ended. Find the rides where it didn't (staff moved the bike): bike_id, ride_id, expected_station (where the previous ride ended) and actual_station (where this ride started). Sort by bike_id, then ride_id.

Expected output:

```
+-------+-------+----------------+--------------+
|bike_id|ride_id|expected_station|actual_station|
+-------+-------+----------------+--------------+
|     B1|      7|               1|             2|
|     B2|      5|               4|             3|
|     B3|     13|               6|             1|
+-------+-------+----------------+--------------+
```

In [ ]:
def etl(rides):
    # Write code here: return a DataFrame
    return rides

etl(rides).show()

**Q16 · `etl(rides)`** (hard) What is each member's longest streak of consecutive days with at least one ride? Show member_id and longest_streak, sorted by member_id.

Expected output:

```
+---------+--------------+
|member_id|longest_streak|
+---------+--------------+
|        1|             3|
|        2|             2|
|        3|             4|
|        4|             1|
+---------+--------------+
```

In [ ]:
def etl(rides):
    # Write code here: return a DataFrame
    return rides

etl(rides).show()

## Optimization checklist

What Days 13 to 15 taught, as questions to ask of any slow job (and in interviews):

| Ask | Look at | Fix |
|---|---|---|
| What does Spark actually run? | `explain()`, the SQL / DataFrame tab | read it bottom-up; count the `Exchange`s |
| Does it read only what it needs? | `ReadSchema`, `PushedFilters`, `PartitionFilters`, `read` in `measure` | select columns; filter on raw columns (no functions around them); partition folders by a filter column; sort the files you write |
| How many shuffles, and of how much? | `Exchange` operators; shuffle write per stage | one `agg` per key; aggregate before joining; no `distinct` or `orderBy` that isn't needed; top N with a filter right after the rank |
| Which join algorithm? | `BroadcastHashJoin` vs `SortMergeJoin`, `BroadcastNestedLoopJoin` | `F.broadcast` small, filtered or aggregated sides; add an equality to range joins |
| Is one task much slower than the others? | median vs max in the Stages tab, `measure(stages=True)` | broadcast, AQE skew join, salting, a separate path for the hot key |
| Does a window have a `partitionBy`? | `Exchange SinglePartition` | always partition; a `groupBy` + broadcast join back for group totals |
| Is Python on the path of every row? | `BatchEvalPython`, `ArrowEvalPython` | built-in functions |
| Is the same thing computed twice? | the same scans and joins twice in the plan | one pass; `cache()` what several actions reuse, then `unpersist()` |
| Does the driver do the work? | `collect`, `toPandas`, `count() > 0`, loops | aggregate in Spark, `isEmpty()`, one `select` |
| Did it actually get faster? | `measure` or the UI, before and after | change one thing at a time |

Official guides to keep: [Performance tuning](https://spark.apache.org/docs/latest/sql-performance-tuning.html), [Tuning Spark](https://spark.apache.org/docs/latest/tuning.html), [Web UI](https://spark.apache.org/docs/latest/web-ui.html), [Configuration](https://spark.apache.org/docs/latest/configuration.html).

## Next: Zillacode

You now know every tool its 52 PySpark problems use, and how Spark runs them. Set it up (the last task of Day 15 in the roadmap), then solve the problems Easy first, then Medium, then Hard. For each one: read the output schema, plan the steps in a comment, write `etl`, and check the "no error, wrong answer" list from Lesson 6 before you press Run. If a problem fails, compare your output with the expected one row by row; the difference almost always points at nulls, ties, types or an extra column.

Zillacode's problems by topic (number and level: E, M, H), so you can drill a weak topic:

| Topic | Problems |
|---|---|
| Complex Joins | 2M, 3H, 5E, 10M, 13H, 14E, 15H, 18M, 20H, 23M, 24H, 27M, 31M, 38E, 46E, 47E, 49M, 52E |
| Aggregate Functions | 3H, 5E, 8E, 10M, 15H, 19M, 20H, 23M, 25E, 27M, 32M, 35E, 36E, 39E, 42E, 44M |
| Windows Functions | 7H, 9M, 13H, 15H, 21H, 22H, 24H, 28M, 29H, 30H, 37H, 43H, 45H |
| Regular Expressions | 4M, 6E, 17E, 26H, 30H, 51H |
| Conditional Logic | 1E, 7H, 32M, 36E, 50E |
| Datetime Operations | 1E, 20H, 22H, 30H, 31M |
| Case When Statements | 16E, 21H, 33M, 41E, 45H |
| String Manipulation | 2M, 21H, 40M, 48M |
| Null Handling | 11E, 14E, 26H |
| Mathematical Functions | 29H, 45H, 51H |
| Distinct Function | 8E, 13H |
| Union Statements | 12E, 34E |
| Pivot Function | 3H |

## Answers
Try each exercise yourself first. Click an exercise to open its answer.

<details>
<summary><b>Q1</b> &nbsp;Split the events (duplicates removed) of customers 1 and 2 into sessions: a new session starts when more than ...</summary>

```python
def etl(events):
    w = W.partitionBy("customer_id").orderBy("event_time")
    gap_minutes = (F.unix_timestamp("event_time") - F.unix_timestamp(F.lag("event_time").over(w))) / 60
    return (events.dropDuplicates()
                  .filter(F.col("customer_id").isin(1, 2))
                  .withColumn("new_session", F.when(gap_minutes.isNull() | (gap_minutes > 30), 1).otherwise(0))
                  .withColumn("session", F.sum("new_session").over(w.rowsBetween(W.unboundedPreceding, W.currentRow)))
                  .select("customer_id", "event_time", "page", "session")
                  .orderBy("customer_id", "event_time"))

etl(events).show()
```

</details>

<details>
<summary><b>Q2</b> &nbsp;What is each customer's longest streak of consecutive days with at least one event? Show customer_id and ...</summary>

```python
def etl(events):
    days = events.select("customer_id", F.to_date("event_time").alias("day")).distinct()
    w = W.partitionBy("customer_id").orderBy("day")
    streaks = (days.withColumn("anchor", F.date_sub("day", F.row_number().over(w)))
                   .groupBy("customer_id", "anchor")
                   .agg(F.count("*").alias("days")))
    return (streaks.groupBy("customer_id")
                   .agg(F.max("days").alias("longest_streak"))
                   .orderBy("customer_id"))

etl(events).show()
```

</details>

<details>
<summary><b>Q3</b> &nbsp;Build the current customer list from two snapshots: customers (taken on 2024-03-31) and customers_new (taken ...</summary>

```python
def etl(customers, customers_new):
    both = (customers.withColumn("snapshot", F.to_date(F.lit("2024-03-31")))
                     .unionByName(customers_new.withColumn("snapshot", F.to_date(F.lit("2024-04-07")))))
    w = W.partitionBy("customer_id").orderBy(F.desc("snapshot"))
    return (both.withColumn("rn", F.row_number().over(w))
                .filter(F.col("rn") == 1)
                .select("customer_id", "name", "city", "snapshot")
                .orderBy("customer_id"))

etl(customers, customers_new).show()
```

</details>

<details>
<summary><b>Q4</b> &nbsp;Solve it with spark.sql: the cheapest product of each category. Show category, product_name and price, sorted ...</summary>

```python
def etl(products):
    products.createOrReplaceTempView("products")
    return spark.sql("""
        SELECT category, product_name, price
        FROM (SELECT category, product_name, price,
                     ROW_NUMBER() OVER (PARTITION BY category ORDER BY price) AS rn
              FROM products)
        WHERE rn = 1
        ORDER BY category
    """)

etl(products).show()
```

</details>

<details>
<summary><b>Q5</b> &nbsp;Which of three December queries reads the fewest bytes, and why does the month-folder table help even without PartitionFilters?</summary>

From the most bytes to the fewest: (a) about 17.6 MB, (b) about 1.4 MB, (c) about 290 KB.

(a) The files are in random date order: every row group's min and max of `sale_date` span the two years, so no row group can be skipped, and both columns the query needs (`sale_date` and `quantity`) are read in full.

(b) There are no `PartitionFilters` (the filter is on `sale_date`, not on the folder column), so Spark lists every folder and opens every file. But each file holds the dates of one month, so the min and max of every row group outside December rule it out: the reader reads its footer and skips its data. The `PushedFilters` on `sale_date` do this work.

(c) `PartitionFilters` on `sale_month`: Spark doesn't even list the other 23 folders; it opens only December's files. Lesson: partition by the column you filter on, and filter on the partition column itself.

</details>

<details>
<summary><b>Q6</b> &nbsp;Add up the items of each channel in the first quarter of 2024 (January to March), sorted by channel. slow_etl ...</summary>

```python
def etl(by_month):
    return (by_month.filter(F.col("sale_month").between("2024-01", "2024-03"))
                    .groupBy("channel").agg(F.sum("quantity").alias("items"))
                    .orderBy("channel"))

etl(by_month).show()
```

`sale_month` is text like `2024-01`, so `between("2024-01", "2024-03")` (or `isin("2024-01", "2024-02", "2024-03")`) compares text, which works because the format sorts like the dates. The slow version wraps `sale_date` in `year()` and `quarter()`: nothing can be pushed, and every folder and file is read. A filter on the raw dates, `F.col("sale_date").between("2024-01-01", "2024-03-31")`, would skip most row groups through their min/max (as in (b) of Q5), but every file's footer is still read; only a filter on the partition column skips whole folders.

</details>

<details>
<summary><b>Q7</b> &nbsp;Which of four programs gain from cache(), and which lose?</summary>

**Gain: (b) and (d).** (b) is the ideal case: an expensive pass over 10 million rows that comes out small (at most 600 rows) and is reused five times. (d) reuses the expensive result 24 times; without the cache, each write would redo the joins and windows (also consider `clean.write.partitionBy("month")`, a single write that does it in one pass).

**Lose: (a) and (c).** (a) uses the DataFrame once: the cache only adds the cost of storing 10 million joined rows with every column. (c) stores all 10 million sales, every column, to show 20 rows of one store; and the filter can no longer be pushed into the Parquet scan (Lesson 4's last cell). `sales.filter(...).show()` alone reads far less.

</details>

<details>
<summary><b>Q8</b> &nbsp;Count the weekend sales (Saturday or Sunday) of each channel, sorted by channel. slow_etl does it with a ...</summary>

```python
def etl(sales):
    return (sales.filter(F.dayofweek("sale_date").isin(1, 7))
                 .groupBy("channel").agg(F.count("*").alias("weekend_sales"))
                 .orderBy("channel"))

etl(sales).show()
```

Spark's `dayofweek` counts from Sunday = 1 to Saturday = 7, so the weekend is `isin(1, 7)` (Python's `weekday()` counts from Monday = 0). The UDF version sends 10 million dates to Python and back; the built-in one never leaves the JVM's generated code.

</details>

<details>
<summary><b>Q9</b> &nbsp;Find four problems in daily_report() without running it.</summary>

1. `sales.count() == 0` counts 10 million rows to answer yes or no: use `sales.isEmpty()`.
2. `report.collect()` brings 10 million joined rows to the driver, which is slow and can crash it with an out-of-memory error.
3. The Python loop adds up the items per city on the driver. Spark should do it: `sales.join(...).groupBy("city").agg(F.sum("quantity"))` returns 10 rows.
4. `orderBy("sale_date")` sorts 10 million rows that nobody needs in order: a full shuffle and sort for nothing.

Also: the three `is_...` columns are never used (harmless here, since Catalyst drops unused columns, but noise), and the join reads all of `customers_big` when only `city` is needed. Three `withColumn` calls in a loop are fine; it's hundreds of them that hurt. The whole function is one query: `sales.join(customers_big, "customer_id").groupBy("city").agg(F.sum("quantity"))`.

</details>

<details>
<summary><b>Q10</b> &nbsp;For 2024, show for each city: sales (the number of sales), items, customers (the number of different ...</summary>

```python
def etl(sales, customers_big):
    weekend = F.dayofweek("sale_date").isin(1, 7)
    return (sales.filter(F.col("sale_date").between("2024-01-01", "2024-12-31"))
                 .join(F.broadcast(customers_big), "customer_id")
                 .groupBy("city")
                 .agg(F.count("*").alias("sales"),
                      F.sum("quantity").alias("items"),
                      F.countDistinct("customer_id").alias("customers"),
                      F.sum(F.when(weekend, 1).otherwise(0)).alias("weekend_sales"))
                 .withColumn("weekend_pct", F.round(F.col("weekend_sales") / F.col("sales") * 100, 1))
                 .select("city", "sales", "items", "customers", "weekend_pct")
                 .orderBy("city"))

etl(sales, customers_big).show()
```

The four problems of `slow_etl`:
1. **A Python UDF** for the weekend: `ArrowEvalPython` in the plan, every 2024 sale sent to Python. `F.dayofweek(...).isin(1, 7)` stays in the JVM (Lesson 5).
2. **`F.year("sale_date") == 2024`** wraps the column, so `PushedFilters` has no date (Day 13, Lesson 2). On these random-order files it reads the same bytes either way, but on files sorted or partitioned by date (Lesson 3) it decides how much is read.
3. **The same joined data aggregated three times and joined back** (Day 13, Lesson 5): one `groupBy` with all four aggregates does it in one pass; the weekend count becomes a conditional sum.
4. **A `cache()` that only exists because of problem 3**, and it makes things worse: the cache must hold every column of the join, so column pruning stops there. `customers_big` is then needed with all four columns, about 18 MB, which is over the broadcast limit: a sort-merge join shuffles 5 million sales. Without the cache, only `customer_id` and `city` are needed (about 8 MB), and the join is broadcast (`F.broadcast` makes it explicit).

`countDistinct` still costs a second shuffle (by city and customer); if an estimate within a few percent were acceptable, `approx_count_distinct` would remove it, but the task asks for the exact number.

</details>

<details>
<summary><b>Q11</b> &nbsp;(easy) Which rides lasted more than 30 minutes? Show ride_id, member_id and minutes (a whole number), sorted ...</summary>

```python
def etl(rides):
    minutes = ((F.unix_timestamp(F.to_timestamp("end_time")) - F.unix_timestamp(F.to_timestamp("start_time"))) / 60).cast("int")
    return (rides.select("ride_id", "member_id", minutes.alias("minutes"))
                 .filter(F.col("minutes") > 30)
                 .orderBy("ride_id"))

etl(rides).show()
```

</details>

<details>
<summary><b>Q12</b> &nbsp;(easy) Which members have never taken a ride? Show member_id and full_name.</summary>

```python
def etl(members, rides):
    return members.join(rides, "member_id", "left_anti").select("member_id", "full_name").orderBy("member_id")

etl(members, rides).show()
```

</details>

<details>
<summary><b>Q13</b> &nbsp;(medium) Show every ride with the names of its stations: ride_id, from_station and to_station (empty when the ...</summary>

```python
def etl(rides, stations):
    start = stations.select(F.col("station_id").alias("start_station"), F.col("station_name").alias("from_station"))
    end = stations.select(F.col("station_id").alias("end_station"), F.col("station_name").alias("to_station"))
    return (rides.join(start, "start_station", "left")
                 .join(end, "end_station", "left")
                 .select("ride_id", "from_station", "to_station")
                 .orderBy("ride_id"))

etl(rides, stations).show()
```

</details>

<details>
<summary><b>Q14</b> &nbsp;(medium) Find each member's favourite start station: the one they started the most rides from (on a tie, the ...</summary>

```python
def etl(rides, stations):
    counts = rides.groupBy("member_id", "start_station").agg(F.count("*").alias("rides"))
    w = W.partitionBy("member_id").orderBy(F.desc("rides"), "start_station")
    return (counts.withColumn("rn", F.row_number().over(w))
                  .filter(F.col("rn") == 1)
                  .join(stations, F.col("start_station") == F.col("station_id"))
                  .select("member_id", "station_name", "rides")
                  .orderBy("member_id"))

etl(rides, stations).show()
```

</details>

<details>
<summary><b>Q15</b> &nbsp;(hard) A bike should start each ride where its previous ride ended. Find the rides where it didn't (staff ...</summary>

```python
def etl(rides):
    w = W.partitionBy("bike_id").orderBy(F.to_timestamp("start_time"))
    return (rides.withColumn("expected_station", F.lag("end_station").over(w))
                 .filter(F.col("expected_station") != F.col("start_station"))
                 .select("bike_id", "ride_id", "expected_station", F.col("start_station").alias("actual_station"))
                 .orderBy("bike_id", "ride_id"))

etl(rides).show()
```

Order by the start time, not by ride_id: ride 13 started before ride 12, and ordering bike B3's rides by id gives a wrong answer. The first ride of each bike has a null `expected_station`, and `null != x` is null, so the filter drops it without an extra test.

</details>

<details>
<summary><b>Q16</b> &nbsp;(hard) What is each member's longest streak of consecutive days with at least one ride? Show member_id and ...</summary>

```python
def etl(rides):
    days = rides.select("member_id", F.to_date("start_time").alias("day")).distinct()
    w = W.partitionBy("member_id").orderBy("day")
    return (days.withColumn("anchor", F.date_sub("day", F.row_number().over(w)))
                .groupBy("member_id", "anchor")
                .agg(F.count("*").alias("days"))
                .groupBy("member_id")
                .agg(F.max("days").alias("longest_streak"))
                .orderBy("member_id"))

etl(rides).show()
```

</details>